<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/06_matmatp/06_matmatp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `06_matmatp` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/06_matmatp
!ls


# Blocking in matrix-matrix products on dense matrices
* Author:       Yukihiro Ota (yota@rist.or.jp)
* Last update:  26th Jan. 2024

## Instruction: Compile
1. Source code is saved in `src/`. Choose either fortran or c.
2. Change directory.

In [ ]:
!cd src/c # On C


3. Before compiling the code, set `NSIZE` (matrix dimension) in `main.F90` or `main.c`.  
  * Before running this sample, we recommend that you measure the memory latency in your machine using [LMbench](https://lmbench.sourceforge.net/). You can estimate which level of cache is almost equivalent to the main memory from a latency point of view.   
  * `LMbench` requires `libtirpc`. If the library and the related header file are absent in your machine, you need to obtain it. On RHEL8, the corresponding library is involved in `libtirpc-devel`.
4. Type `make` with your desired compiler setting.

In [ ]:
!make


The code is successfully compiled by
   * GNU (8.5.0) on x86-64 systems 


## Instruction: Run and do analyses
1. Sample scripts are stored in `tests/`. Choose either fortran or c.       
2. Change directory

In [ ]:
!cd tests/c


3. Run a job script, `run.sh`.

In [ ]:
# One example
!bash run.sh
# Another example
!chmod 755 run.sh
!./run.sh


4. The results will be summarized in files, `output.40x2` and `output.60x2`. The suffixes, e.g., `40x2`, indicate the setting of the two input parameters, `NA` and `NB`; `40x2` means that `NA*NB ~ 40x2=1600`. 

## Exercise 
1. Check your CPU, about the number of processor cores and the number of sockets. The following Linux commands would be helpful.

In [ ]:
!cat /proc/cpuinfo
!numactl -H


2. Check the size of cache with different levels, such as L1, L2, and last-level cache (e.g., L3), in your machine. On Linux, typing `getconf -a`, you can find the corresponding data in `LEVEL2_CACHE_SIZE`, for example. Also, check whether a certain level of cache is shared between cores or not. 
3. Evaluate the cache size per core. Also, set `NSIZE` parameter. Please read the comment on the top of `main.F90` or `main.c`. 
4. Check the access pattern of arrays in `mykernel.F90` or `mykernel.c`.  
5. Examine the elapsed time in each of kernels (`mmp_simple`, `mmp_simple_blk`, and `mmp_lex_tp_blk`), varying the size of blocks. Your compiler may generate better code even in the case of `mmp_simple`. This is a good news if so; It means that you do not need to take care of by-hand optimization. 

## Advanced topics
1. Use a matrix-matrix product routine in a well-tuned library, e.g., DGEMM in BLAS (such as OpenBLAS). Compare the results to those in the hand-made kernels.